# 01 — Data Ingestion & Feature Engineering
PULSE-OPS | End-to-end data pipeline walkthrough.

Covers:
- Fetching UCI Adult, Wine Quality, Bike Sharing, German Credit datasets
- Feature engineering and encoding
- Feature store materialization
- Data quality validation

In [ ]:
import sys
from pathlib import Path

# Ensure repo root is on path
repo_root = Path("__file__").resolve().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

print("Environment ready")

## 1. Fetch Datasets

In [ ]:
from data.fetchers.uci_fetcher import UCIFetcher
from data.fetchers.bike_fetcher import BikeSharingFetcher
from data.fetchers.credit_fetcher import GermanCreditFetcher

uci = UCIFetcher()
adult_df = uci.fetch_adult_dataset()
wine_df = uci.fetch_wine_quality()

bike_fetcher = BikeSharingFetcher()
bike_df = bike_fetcher.load_hourly()

credit_fetcher = GermanCreditFetcher()
credit_df = credit_fetcher.download_and_parse()

print(f"Adult:   {adult_df.shape}")
print(f"Wine:    {wine_df.shape}")
print(f"Bike:    {bike_df.shape}")
print(f"Credit:  {credit_df.shape}")

## 2. Feature Engineering

In [ ]:
from data.processors.feature_engineer import FeatureEngineer

engineer = FeatureEngineer()

# Adult dataset
adult_clean = engineer.handle_missing(adult_df.copy())
adult_encoded = engineer.encode_categoricals(adult_clean)
adult_ts = engineer.add_event_timestamp(adult_encoded)

print("Adult after encoding:")
print(adult_encoded.dtypes.value_counts())
print(f"\nShape: {adult_encoded.shape}")
adult_encoded.head(3)

In [ ]:
# Wine dataset — purely numeric, no encoding needed
wine_clean = engineer.handle_missing(wine_df.copy())
print("Wine missing values:", wine_clean.isnull().sum().sum())
print("Quality distribution:")
print(wine_clean["quality"].value_counts().sort_index())

## 3. Data Splitting

In [ ]:
from data.processors.data_splitter import DataSplitter

splitter = DataSplitter()
X_train, X_val, X_test, y_train, y_val, y_test = splitter.split(adult_encoded, "income")

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Target distribution (train): {y_train.value_counts(normalize=True).round(3).to_dict()}")

## 4. Data Validation

In [ ]:
from data.processors.data_validator import DataValidator

validator = DataValidator()
results = validator.validate(adult_encoded)

for check_name, check_result in results.items():
    status = "✓" if check_result["passed"] else "✗"
    print(f"  {status} {check_name}: {check_result}")

## 5. Feature Store Materialization

In [ ]:
from feature_store.feature_pipeline import FeaturePipeline
from feature_store.offline_store import OfflineStore
from feature_store.online_store import OnlineStore

fp = FeaturePipeline()

# Save features to parquet (offline store)
raw_dir = Path("__file__").resolve().parent.parent / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
adult_ts.to_parquet(raw_dir / "adult_features.parquet", index=False)
print(f"Saved {len(adult_ts)} records to offline store")

offline = OfflineStore()
hist_df = offline.get_historical_features("adult", [])
print(f"Historical features shape: {hist_df.shape}")

online = OnlineStore()
record = online.get_online_features("adult", 0)
print(f"Online lookup sample keys: {list(record.keys())[:5]}")

## 6. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Adult — age distribution by income
if "age" in adult_df.columns and "income" in adult_df.columns:
    for label, grp in adult_df.groupby("income"):
        grp["age"].hist(ax=axes[0], alpha=0.6, bins=30, label=str(label))
    axes[0].set_title("Adult: Age Distribution by Income")
    axes[0].set_xlabel("Age")
    axes[0].legend()

# Wine — quality histogram
wine_df["quality"].value_counts().sort_index().plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_title("Wine Quality Score Distribution")
axes[1].set_xlabel("Quality Score")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for wine quality features
import seaborn as sns

fig, ax = plt.subplots(figsize=(10, 8))
corr = wine_clean.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax, square=True)
ax.set_title("Wine Quality — Feature Correlation Matrix")
plt.tight_layout()
plt.show()